In [4]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values / 255.0
y = mnist.target.astype(int).values.reshape(-1,1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Activation functions

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return x > 0

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Network structure

input_size = 784
hidden1 = 128
hidden2 = 64
output_size = 10

# Weight initialization (He initialization for ReLU layers, Xavier for output)

W1 = np.random.randn(input_size, hidden1) * np.sqrt(2. / input_size)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2. / hidden1)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size) * np.sqrt(1. / hidden2) # Xavier initialization
b3 = np.zeros((1, output_size))

learning_rate = 0.001 # Reduced learning rate
epochs = 50 # Increased epochs
batch_size = 64 # Mini-batch size

num_batches = len(X_train) // batch_size

for epoch in range(epochs):
    # Shuffle training data for each epoch
    permutation = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[permutation]
    y_train_shuffled = y_train[permutation]

    epoch_loss = 0

    for i in range(num_batches):
        start = i * batch_size
        end = start + batch_size

        X_batch = X_train_shuffled[start:end]
        y_batch = y_train_shuffled[start:end]

        # Forward propagation
        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)

        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        # Cross entropy loss
        loss = -np.mean(y_batch * np.log(A3 + 1e-8))
        epoch_loss += loss

        # Backpropagation
        dZ3 = A3 - y_batch
        dW3 = np.dot(A2.T, dZ3)
        db3 = np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, W3.T)
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = np.dot(X_batch.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/num_batches}")

# Testing

Z1 = np.dot(X_test, W1) + b1
A1 = relu(Z1)

Z2 = np.dot(A1, W2) + b2
A2 = relu(Z2)

Z3 = np.dot(A2, W3) + b3
A3 = softmax(Z3)

predictions = np.argmax(A3, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == true_labels)

print("Test Accuracy:", accuracy)

Epoch 1, Loss: 0.043572352280448735
Epoch 2, Loss: 0.019869261710667465
Epoch 3, Loss: 0.01482845994563058
Epoch 4, Loss: 0.01195506029548287
Epoch 5, Loss: 0.01002105682693827
Epoch 6, Loss: 0.008446251669726079
Epoch 7, Loss: 0.007330780588770942
Epoch 8, Loss: 0.00634411716520954
Epoch 9, Loss: 0.005547828212845857
Epoch 10, Loss: 0.004954280909053117
Epoch 11, Loss: 0.004330939321102372
Epoch 12, Loss: 0.00389086074970572
Epoch 13, Loss: 0.0034146152160732532
Epoch 14, Loss: 0.0030735733728723155
Epoch 15, Loss: 0.002677604538182416
Epoch 16, Loss: 0.0024282242800102785
Epoch 17, Loss: 0.002030095869440788
Epoch 18, Loss: 0.0018046114535719673
Epoch 19, Loss: 0.0016391063852016126
Epoch 20, Loss: 0.0014081592877439317
Epoch 21, Loss: 0.0012462805879237483
Epoch 22, Loss: 0.0011206516051432318
Epoch 23, Loss: 0.0009620130948685569
Epoch 24, Loss: 0.0008411366909571341
Epoch 25, Loss: 0.0007555795072965169
Epoch 26, Loss: 0.0006675693439413481
Epoch 27, Loss: 0.0005725815153214178
Ep